# 06a — Review sentiment (NLP)

Positive = 4–5★, negative = 1–2★, 3★ dropped. TF-IDF fit on train-period reviews only; evaluated on validation-period reviews.

No stop-word list: sklearn's English list removes "not", "no", "never", which carry sentiment.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

sys.path.insert(0, "../src")
from shoprank.nlp.sentiment import product_sentiment, review_text, sentiment_labels

PROC = Path("../data/processed")
train = pd.read_parquet(PROC / "train.parquet")
val = pd.read_parquet(PROC / "val.parquet")
items = pd.read_parquet(PROC / "items.parquet")

train["y"], val["y"] = sentiment_labels(train), sentiment_labels(val)
tr, va = train[train["y"].notna()], val[val["y"].notna()]
y_tr, y_va = tr["y"].astype(int).to_numpy(), va["y"].astype(int).to_numpy()
print(f"train {len(tr):,} labeled ({y_tr.mean():.1%} positive)   val {len(va):,} labeled ({y_va.mean():.1%} positive)")

train 139,115 labeled (89.3% positive)   val 19,792 labeled (90.8% positive)


## Features: TF-IDF on `title + text`

In [2]:
t = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=200_000, sublinear_tf=True)
X_tr = vec.fit_transform(review_text(tr))
X_va = vec.transform(review_text(va))
print(f"vocabulary {len(vec.vocabulary_):,}, fit in {time.time() - t:.0f}s")

vocabulary 183,464, fit in 7s


## Compare models on validation

Negative reviews are the rare class, so the key numbers are negative-class F1, macro F1 and PR-AUC for finding negatives. LinearSVC has no probabilities; its decision score is used for PR-AUC.

In [3]:
def scores(y_true, y_pred, positive_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "F1 positive": f1_score(y_true, y_pred, pos_label=1),
        "F1 negative": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro F1": f1_score(y_true, y_pred, average="macro"),
        "PR-AUC positive": average_precision_score(y_true, positive_score),
        "PR-AUC negative": average_precision_score(1 - y_true, -positive_score),
    }

models = {
    "Multinomial NB": MultinomialNB(alpha=0.1),
    "Logistic Regression (balanced)": LogisticRegression(class_weight="balanced", C=4.0, max_iter=2000),
    "LinearSVC": LinearSVC(C=0.5),
}

rows = {"Always positive": scores(y_va, np.ones_like(y_va), np.ones(len(y_va)))}
for name, model in models.items():
    t = time.time()
    model.fit(X_tr, y_tr)
    s = model.predict_proba(X_va)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_va)
    rows[name] = scores(y_va, model.predict(X_va), s)
    print(f"{name}: {time.time() - t:.0f}s")

results = pd.DataFrame(rows).T.round(4)
results

Multinomial NB: 0s


Logistic Regression (balanced): 1s


LinearSVC: 1s


,accuracy,F1 positive,F1 negative,macro F1,PR-AUC positive,PR-AUC negative
Always positive,0.9081,0.9518,0.0000,0.4759,0.9081,0.0919
Multinomial NB,0.9572,0.9767,0.7349,0.8558,0.9964,0.8305
Logistic Regression (balanced),0.9569,0.9760,0.7892,0.8826,0.9981,0.8912
LinearSVC,0.9683,0.9827,0.8122,0.8975,0.9981,0.8996


In [4]:
lr = models["Logistic Regression (balanced)"]
words = np.array(vec.get_feature_names_out())
order = np.argsort(lr.coef_[0])
print("most negative:", ", ".join(words[order[:15]]))
print("most positive:", ", ".join(words[order[-15:][::-1]]))

best = results.drop("Always positive")["macro F1"].idxmax()
baseline = results.loc["Always positive"]
for name in models:
    assert results.loc[name, "macro F1"] > baseline["macro F1"], name
print(f"best macro F1: {best}")

most negative: not, not worth, two stars, terrible, garbage, disappointing, boring, worst, poor, horrible, disappointed, one star, not good, bad, useless
most positive: great, perfect, awesome, excellent, amazing, good, best, not bad, love, fantastic, solid, works, excelente, fun, five stars
best macro F1: LinearSVC


## Output: `avg_sentiment` per product

Logistic Regression supplies the positive probability. Predicting on the very reviews a model was fit on is over-confident, so labeled train reviews get **out-of-fold** probabilities (5 folds, vectorizer refit per fold); 3★ train reviews were never trained on and use the full model. Products with no train reviews get the train-wide mean.

In [5]:
t = time.time()
text_tr = review_text(tr).to_numpy()
oof = np.zeros(len(tr))
for fit_idx, pred_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(text_tr, y_tr):
    v = clone(vec).fit(text_tr[fit_idx])
    m = clone(lr).fit(v.transform(text_tr[fit_idx]), y_tr[fit_idx])
    oof[pred_idx] = m.predict_proba(v.transform(text_tr[pred_idx]))[:, 1]
print(f"out-of-fold in {time.time() - t:.0f}s   in-sample mean p {lr.predict_proba(X_tr)[:, 1].mean():.3f} vs out-of-fold {oof.mean():.3f}")

train["p_positive"] = np.nan
train.loc[tr.index, "p_positive"] = oof
three = train["y"].isna()
train.loc[three, "p_positive"] = lr.predict_proba(vec.transform(review_text(train[three])))[:, 1]
print(f"3-star train reviews: mean p_positive {train.loc[three, 'p_positive'].mean():.3f}")

item_sent = items[["parent_asin"]].merge(product_sentiment(train, train["p_positive"].to_numpy()), how="left")
default = train["p_positive"].mean()
item_sent["avg_sentiment"] = item_sent["avg_sentiment"].fillna(default)
item_sent[["sentiment_sum", "sentiment_n"]] = item_sent[["sentiment_sum", "sentiment_n"]].fillna(0)
print(f"products: {len(item_sent):,}, without train reviews (filled with {default:.3f}): {(item_sent['sentiment_n'] == 0).sum():,}")

train[["user_id", "parent_asin", "p_positive"]].to_parquet(PROC / "train_sentiment.parquet", index=False)
item_sent.to_parquet(PROC / "item_sentiment.parquet", index=False)
item_sent["avg_sentiment"].describe().round(3)

out-of-fold in 51s   in-sample mean p 0.848 vs out-of-fold 0.846


3-star train reviews: mean p_positive 0.491
products: 9,999, without train reviews (filled with 0.822): 1,480


count    9999.000
mean        0.818
std         0.131
min         0.004
25%         0.767
50%         0.822
75%         0.912
max         1.000
Name: avg_sentiment, dtype: float64

In [6]:
check = item_sent.merge(items[["parent_asin", "train_avg_rating", "train_rating_count"]])
check = check[check["train_rating_count"] >= 5]
print(f"correlation of avg_sentiment with train_avg_rating (products with >=5 train reviews): "
      f"{check['avg_sentiment'].corr(check['train_avg_rating']):.3f}")

correlation of avg_sentiment with train_avg_rating (products with >=5 train reviews): 0.873
